<a href="https://colab.research.google.com/github/tu25003-7491-cyber/EU_M_Math-Repository/blob/main/Chap08_Cm_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import numpy.random as random
import scipy as sp
from pandas import Series, DataFrame
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
%matplotlib inline

import sklearn

%precision 3

'%.3f'

In [ ]:
import requests, zipfile
import io

url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content
auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)

auto.columns = [
    'symboling',
    'normalized-losses',
    'make',
    'fuel-type',
    'aspiration',
    'num-of-doors',
    'body-style',
    'drive-wheels',
    'engine-location',
    'wheel-base',
    'length',
    'width',
    'height',
    'curb-weight',
    'engine-type',
    'num-of-cylinders',
    'engine-size',
    'fuel-system',
    'bore',
    'stroke',
    'compression-ratio',
    'horsepower',
    'peak-rpm',
    'city-mpg',
    'highway-mpg',
    'price'
]
auto.head()

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450


In [ ]:
# 指定されたカラム（price, horsepower, width, height）のみを抽出して表示
auto= auto[['price', 'horsepower', 'width', 'height']]
auto.head()

,price,horsepower,width,height
0,13495,111,64.1,48.8
1,16500,111,64.1,48.8
2,16500,154,65.5,52.4
3,13950,102,66.2,54.3
4,17450,115,66.4,54.3


In [ ]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import train_test_split

auto = auto.replace('?', np.nan).dropna()
auto = auto.apply(pd.to_numeric)

x = auto.drop('price', axis=1)
y = auto['price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.5, random_state=0)

linear = LinearRegression()
ridge = Ridge(random_state=0)

for model in [linear, ridge]:
  model.fit(x_train, y_train)
  print("{}(train)): {:.6f}".format(model.__class__.__name__, model.score(x_train, y_train)))
  print("{}(test)): {:.6f}".format(model.__class__.__name__, model.score(x_test, y_test)))


LinearRegression(train)): 0.733358
LinearRegression(test)): 0.737069
Ridge(train)): 0.733355
Ridge(test)): 0.737768


In [ ]:
from sklearn.linear_model import Lasso

for alpha in [0.1, 1.0, 10, 100, 1000]:
    lasso = Lasso(alpha=alpha, random_state=0)
    lasso.fit(x_train, y_train)

    print(f"--- Alpha: {alpha} ---")
    print("Lasso(train): {:.6f}".format(lasso.score(x_train, y_train)))
    print("Lasso(test): {:.6f}".format(lasso.score(x_test, y_test)))
    print("Lasso coefficients:", lasso.coef_)
    print()

--- Alpha: 0.1 ---
Lasso(train): 0.733358
Lasso(test): 0.737073
Lasso coefficients: [  81.653 1829.136  229.505]

--- Alpha: 1.0 ---
Lasso(train): 0.733358
Lasso(test): 0.737107
Lasso coefficients: [  81.668 1828.797  229.46 ]

--- Alpha: 10 ---
Lasso(train): 0.733357
Lasso(test): 0.737416
Lasso coefficients: [  81.807 1825.598  228.949]

--- Alpha: 100 ---
Lasso(train): 0.733288
Lasso(test): 0.740234
Lasso coefficients: [  83.088 1795.04   223.432]

--- Alpha: 1000 ---
Lasso(train): 0.726472
Lasso(test): 0.762360
Lasso coefficients: [  95.899 1489.461  168.258]



### リッジ回帰とラッソ回帰における正則化項の目的とメリット

リッジ回帰とラッソ回帰は、通常の線形回帰に正則化項を加えることで、モデルの係数が極端に大きくなることを防ぐ手法である。通常の線形回帰では、学習データに対して誤差を小さくすることだけを目的とするため、説明変数どうしの相関が強い場合や、データ数に対して変数が多い場合に、係数が不安定になりやすい。このような問題を抑えるために、正則化項を加える。


リッジ回帰では、係数の二乗和を罰則として加える。これにより、大きな係数が抑えられ、モデルが特定の説明変数に過度に依存しにくくなる。特に、多重共線性がある場合、通常の線形回帰では係数の値が大きく変動しやすいが、リッジ回帰では係数を全体的に小さくすることで推定を安定させることができる。ただし、リッジ回帰では係数は小さくなるが、基本的に0にはなりにくいため、変数選択の効果は弱い。


ラッソ回帰では、係数の絶対値の和を罰則として加える。ラッソ回帰の特徴は、一部の係数を0にすることがある点である。そのため、重要度の低い説明変数を自動的に除外するような働きがあり、パラメーター選択や特徴量選択に利用できる。ただし、相関の強い説明変数が複数ある場合には、その中から一部の変数だけを選ぶことがあり、どの変数が選ばれるかはデータの分割や正則化の強さに影響される。


正則化の大きさは、alpha などのパラメーターで調整される。alpha が小さい場合は通常の線形回帰に近くなり、正則化の効果は弱い。一方で alpha が大きすぎると係数が小さくなりすぎ、モデルが単純になりすぎる。そのため、学習データへの当てはまりだけでなく、テストデータに対する性能も確認しながら、適切な alpha を選ぶ必要がある。一般には、交差検証を用いて alpha を選択する。


以上より、リッジ回帰とラッソ回帰の正則化項には、係数の過度な増大を防ぎ、多重共線性による不安定性を抑え、過学習を軽減する目的がある。特に、リッジ回帰は係数を安定させることに有効であり、ラッソ回帰は不要な変数を除くパラメーター選択に有効である。